# Capstone — Honest Machine Learning & Decision-Support for Content Refresh Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

### Abstract
Prioritizing content inventory for manual editorial refresh under finite operational capacity requires identifying pages experiencing real performance decline without wasting review bandwidth on healthy URLs. Using an anonymized 30,000-page dataset across 32 enterprise clients alongside a 78.8M-row daily performance warehouse from FlyRank, we evaluated rule-based heuristics, current-window classifiers, and forward-looking time-series predictive models. In client-holdout validation on the 30K dataset, Logistic Regression achieved a Precision@20 of 0.700 and Precision@50 of 0.600 against a base decline rate of 0.511, outperforming a volume-weighted baseline rule (Precision@20 = 0.400, Precision@50 = 0.420). However, when evaluating a forward-looking predictive model on the full warehouse, honest time-aware validation caused accuracy to collapse from 0.669 (naive random split) to 0.274, exacerbated by severe autocorrelation (r = 0.819) between historical position and the target outcome. Consequently, we rejected the predictive model for deployment, implementing instead an evidence-backed, transparent action playbook with structured reason codes (`STALE_LOW_CTR` and `PAGE1_LOW_CTR`) to govern human-in-the-loop content refresh queues safely.

## 1. Question

**The Decision:** Which pages in a client's content inventory should be reviewed first for refresh or optimization, given limited weekly editorial bandwidth (typically 20–50 pages per cycle)?

**Who Acts:** Content strategists and SEO editors who must triage large content inventories under strict time constraints.

**Cost of a Wrong Call:**
- *False Positive:* Wastes valuable editorial hours reviewing high-performing or stable pages that need no intervention.
- *False Negative:* Allows decaying high-potential pages to lose organic search visibility silently until ranking recovery becomes significantly more expensive.

Because operational review capacity is fixed and small, the primary evaluation metric is **Precision@K** (specifically K=20 and K=50) rather than overall accuracy or ROC-AUC.

In [ ]:
# Setup environment
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Check dataset availability
data_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in data_candidates if os.path.exists(p)), "data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Starter dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Base decline rate (trend_direction == 'down'):", round(df["is_declining"].mean(), 3))


## 2. Data

This research spans two data releases provided in the FlyRank ML Internship:
1. **Starter Anonymized Dataset (`content_refresh_anonymized.csv`):** 30,000 pseudonymized page records across 32 clients. Each record aggregates 90-day Google Search Console (GSC) and Google Analytics 4 (GA4) performance metrics alongside content age and metadata.
2. **Full Data Warehouse Release (`FlyRank/internship-warehouse`):** 78.8M daily performance records (`fact_content_daily_performance`, 28.9M with active GSC impressions) joined with content dimension tables (`dim_content`). Used for longitudinal time-aware validation across distinct calendar windows (Feb–May 2026) and warehouse-scale action queue generation (March 2026 snapshot).

**Exclusions and Sanitization:**
- Filtered to established pages with non-zero visibility: `impressions_90d > 0` and `content_age_days >= 90`.
- Strictly pseudonymized: zero client names, domains, URLs, titles, keywords, or raw query text.
- All metric percentages (`ctr`, `engagement_rate`, `scroll_rate`) are standardized on a 0–100 scale (e.g., `ctr = 0.76` represents 0.76%).

In [ ]:
# Inspect data properties and verify volume/age constraints
print(f"Rows meeting minimum age (>=90d) and impressions (>0): {len(df)} / 30000")

# Summary of key observable features
feature_summary = df[["impressions_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days", "word_count"]].describe().T[["mean", "std", "min", "50%", "max"]]
print(feature_summary)


## 3. Methodology

### 3.1 Task Formulation & Label Definitions
- **Current-Window Classification (ML-08 / Starter CSV):** Binary label `is_declining_label = (trend_direction == "down")` (1 = declining, 0 = stable/up). Target represents whether impressions in the trailing 30 days dropped >20% compared to days 31–60 back.
- **Forward-Looking Target (ML-09 / Warehouse):** Binary label `declined = (future_avg_position > avg_position)` measured across distinct consecutive monthly windows (features in month $T$, label evaluated in month $T+1$).

### 3.2 Baseline Heuristic Rule
A transparent domain rule combining search position expectations and traffic scale:
$$\text{Baseline Score} = \mathbb{I}(\text{CTR} < \overline{\text{CTR}}_{\text{tier}}) \times \mathbb{I}(\text{Impressions}_{90d} \ge 500) \times \text{Impressions}_{90d}$$

### 3.3 Validation Design & Leakage Auditing
- **Grouped Client-Holdout Split:** To avoid memorizing client-specific baseline traffic, the 32 clients were partitioned via `GroupShuffleSplit(test_size=0.2, random_state=42)` into 25 training clients (23,837 pages) and 7 test clients (6,163 pages). No client appears in both sets.
- **Time-Aware Out-of-Time Validation:** Models trained on Feb$\to$Mar 2026 outcomes were validated on separate Apr$\to$May 2026 outcomes to test temporal stability.
- **Honest Leakage Reporting:** Note that while no dedicated standalone leakage-check notebook was completed at the ML-05/ML-06 stage, inline feature auditing in ML-08 verified no label-derived fields were used. Crucially, longitudinal auditing in ML-09 uncovered a severe near-leakage autocorrelation ($r = 0.819$) between `avg_position` and future ranking changes.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
features = [
    "impressions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "sessions_90d", "engagement_rate"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values
groups = df["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train set: {len(train_idx)} rows, {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test set:  {len(test_idx)} rows, {df.iloc[test_idx]['client_id'].nunique()} clients (Base rate = {y_test.mean():.3f})")
overlap = set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])
print(f"Client overlap between train/test: {len(overlap)}")


## 4. Results (vs baseline)

### 4.1 ML-08: Current-Window Model Comparison (Starter Dataset)
On the client-holdout test split (6,163 rows, 7 clients, base decline rate = 0.511):
- **Baseline Rule:** Precision@20 = 0.400, Precision@50 = 0.420 (underperforms base rate due to volume-weighting bias).
- **Logistic Regression:** Precision@20 = 0.700, Precision@50 = 0.600 (+37.0% lift over baseline at P@50).
- **Random Forest:** Precision@20 = 0.500, Precision@50 = 0.580.

Logistic regression coefficients indicated that CTR had the strongest negative association with decline ($w = -0.0528$), while Random Forest placed heavy importance on `impressions_90d` (0.303) and `avg_position` (0.228).

### 4.2 ML-09: Warehouse Validation Collapse & Leakage Discovery
Evaluating the forward-looking model on the 78.8M-row warehouse revealed a critical failure mode:
- **Naive Random Split (Time-Mixed):** Accuracy = 0.669 (seemingly viable).
- **Honest Time-Aware Split (Feb$\to$Mar train, Apr$\to$May test):** Accuracy collapsed to **0.274** (substantially below majority-class guessing).
- **Autocorrelation & Near-Leakage Finding:** Audit revealed that `avg_position` correlated with the outcome variable `future_avg_position` at **$r = 0.819$**. Because position is heavily autocorrelated month-over-month and mechanically limits headroom for further decline, the model learned a fragile shortcut that failed under temporal distribution shifts.
- **Architectural Decision:** The predictive forward-looking model was **killed** and rejected from deployment.

### 4.3 ML-10: Deployed Action Playbook
In place of the failed predictive model, a re-validated rule-based prioritization queue was deployed to warehouse data (`month=2026-03`), yielding 59 high-priority refresh candidates categorized by transparent reason codes: `STALE_LOW_CTR` (47 pages, 79.7%) and `PAGE1_LOW_CTR` (12 pages, 20.3%).

In [ ]:
# Reproduce model vs baseline evaluation metrics
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

df_test = df.iloc[test_idx].copy()
pos_tier = pd.cut(df_test["avg_position"], bins=[0, 3, 10, 20, 1000])
tier_avg_ctr_test = df_test.groupby(pos_tier, observed=False)["ctr"].transform("mean")
df_test["below_tier_avg_ctr"] = (df_test["ctr"] < tier_avg_ctr_test).astype(int)
df_test["visible"] = (df_test["impressions_90d"] >= 500).astype(int)
baseline_score_test = df_test["below_tier_avg_ctr"] * df_test["visible"] * df_test["impressions_90d"]

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED, class_weight="balanced")
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

results = []
for name, scores in [("baseline_rule", baseline_score_test), ("logistic_regression", logreg_proba), ("random_forest", rf_proba)]:
    row = {"method": name}
    for k in (20, 50):
        row[f"precision@{k}"] = precision_at_k(scores, y_test, k)
    results.append(row)

results_df = pd.DataFrame(results)
results_df["base_rate"] = y_test.mean()
print(results_df)


## 5. Limitations

1. **Observational, Non-Causal Associations:** All findings reflect measured historical correlations in Search Console and Analytics data. Ranking improvements cannot be guaranteed purely by executing a refresh without active experimentation (e.g., randomized A/B content tests).
2. **No Longitudinal Validation on Current-Window Model:** The ML-08 model was validated on client-holdout partitions within a single 90-day window. Given the failure observed in ML-09, current-window classifiers should not be assumed temporally invariant without explicit longitudinal verification.
3. **Small Client Cohort in Starter Slice:** The 30,000-page starter dataset represents 32 enterprise clients. While grouped holdout prevents client memorization, cohort-level domain bias remains possible.
4. **Granularity Disconnect (Starter CSV vs. Full Warehouse):** The starter dataset provides 90-day pre-aggregated snapshots, whereas the warehouse provides daily granularity. Aggregated rates smooth out short-term volatility but obscure intra-window decay trajectories.
5. **Fixed Feature Set & Lack of Qualitative Context:** Features are restricted to numerical metrics. Semantic content depth, query intent shifts, competitor updates, and technical SEO issues cannot be observed from these tabular vectors.
6. **High Position Autocorrelation ($r=0.819$):** Page ranking positions exhibit strong temporal persistence, making delta-based predictive targets prone to mechanical artifacts.

In [ ]:
# Error analysis: Inspect false positives in top-20 predictions
test_results = df_test.copy()
test_results["pred_proba"] = logreg_proba
test_results["actual"] = y_test
top20_idx = np.argsort(-logreg_proba)[:20]
top20_results = test_results.iloc[top20_idx]

wrong_cases = top20_results[top20_results["actual"] == 0]
fp_count = len(wrong_cases)
precision_20 = (20 - fp_count) / 20.0
print(f"{fp_count} false positive cases in top 20 by Logistic Regression (Precision@20 = {precision_20:.2f}):")
print(wrong_cases[["content_id", "impressions_90d", "avg_position", "ctr", "content_age_days", "pred_proba", "actual"]].head(5))


## 6. Ranked recommendations

### 6.1 Deployed Action Queue & Priority Formula
Because predictive modeling failed out-of-time stability, recommendations are governed by the transparent, re-validated action formula:
$$\text{Priority Score} = \frac{\text{Avg Impressions} \times (\text{Days Since Update} / 365)}{\text{Avg CTR} + 0.005}$$

Filtered to established pages (`days_since_update >= 90`, `avg_impressions >= 20`, `is_deleted = False`). Applied to the March 2026 warehouse partition, this generated a 59-page actionable queue:
- **`STALE_LOW_CTR` (47 pages / 79.7%):** High-decay-risk URLs with extended staleness and depressed click-through performance.
- **`PAGE1_LOW_CTR` (12 pages / 20.3%):** High-visibility URLs ranking in positions 1–10 that capture below-average CTR (<0.50%), representing immediate headline/meta optimization opportunities.

### 6.2 Editorial Review Protocol
Before executing any content modification, the human reviewer must:
1. **Validate Relevance:** Confirm the page aligns with current strategic offerings and is not marked for deprecation or consolidation.
2. **Check Ongoing Edits:** Verify the page is not already undergoing scheduled updates.
3. **Isolate External Factors:** Check whether recent dips reflect seasonal demand shifts or platform-wide SERP layout changes.
4. **Review Data Completeness:** Verify client tracking integrity for low-traffic entries.

### 6.3 Operational No-Go List
- **DO NOT** auto-publish AI-refreshed text without human editorial sign-off.
- **DO NOT** auto-delete or de-index flagged URLs.
- **DO NOT** make revenue or traffic-lift guarantees to clients based on rank priority scores.
- **DO NOT** deploy or automate outputs from the uncalibrated ML-09 predictive model.

In [ ]:
# Load metrics and queue verification
metrics_receipt = {
    "n_ranked": 59,
    "reason_code_counts": {"STALE_LOW_CTR": 47, "PAGE1_LOW_CTR": 12},
    "month_used": "2026-03",
    "signal1_staleness_verdict": "MIXED",
    "signal2_ctr_position_verdict": "CONFIRMED",
    "predictive_model_status": "FAILED_HONEST_VALIDATION",
    "predictive_model_note": "Accuracy dropped 0.669 -> 0.274 under time-aware split; excluded from this playbook"
}
print("Playbook Metrics Receipt:")
print(json.dumps(metrics_receipt, indent=2))


## 7. Artifacts the paper embeds

The four core visual artifacts generated from experimental logs and warehouse audits:

In [ ]:
# Verify all 4 chart figures exist
fig_paths = [
    "work/figures/ml08_model_precision_comparison.png",
    "work/figures/ml09_validation_audit_drop.png",
    "work/figures/ml09_leakage_correlation.png",
    "work/figures/w07_reason_code_breakdown.png"
]

for p in fig_paths:
    abs_p = p if os.path.exists(p) else os.path.join("../..", p)
    if os.path.exists(abs_p):
        print(f"Found artifact: {abs_p}")
    else:
        print(f"Warning: {abs_p} not found")


## 8. ML-12 Deliverables

### 8.1 5-Minute Technical Demo Outline
1. **Minute 1 — The Problem & Decision Gap:** Explain the editorial triage bottleneck: content teams manage thousands of URLs but can only refresh 20–50 pages per cycle. Show why hand-written rules fall into volume traps.
2. **Minute 2 — Current-Window Model vs Baseline:** Present the ML-08 client-holdout evaluation (Precision@50 = 0.600 for LogReg vs. 0.420 baseline). Show why Logistic Regression provided the most robust interpretable ranking.
3. **Minute 3 — The Out-of-Time Reality Check:** Walk through the ML-09 warehouse audit. Show how the naive 66.9% accuracy collapsed to 27.4% under strict temporal splitting due to the 0.819 position autocorrelation.
4. **Minute 4 — Responsible Engineering & The Action Playbook:** Explain why killing the predictive model was the only responsible engineering decision. Present the deployed 59-page action queue and reason-code taxonomy (`STALE_LOW_CTR` vs `PAGE1_LOW_CTR`).
5. **Minute 5 — Operational Safeguards & Next Steps:** Detail the monitoring triggers (May 2026 CTR stability check) and the strict human-in-the-loop review policy.

### 8.2 Social Post Summary
How do you triage 30K+ pages for SEO refresh without burning editorial time? In our latest research on the FlyRank dataset, Logistic Regression delivered a 37% lift in Precision@50 (0.600 vs 0.420 baseline) on held-out clients. But when we scaled to 78.8M warehouse records, time-aware validation caused a forward-looking predictive model to collapse from 66.9% to 27.4% accuracy due to severe ranking autocorrelation (r=0.819). We killed the model and shipped an evidence-backed, transparent action queue instead. Read the full paper: https://eimanzahra1472.github.io/flyrank-ml-internship-starter/

### 8.3 3-Sentence Employer Summary
Built and audited a machine learning content prioritization system across a 30,000-page enterprise slice and a 78.8M-row performance warehouse from FlyRank. Discovered a critical out-of-time generalization failure (accuracy dropping from 66.9% to 27.4%) caused by target autocorrelation ($r=0.819$), leading to a deliberate architectural decision to kill the predictive model rather than ship an uncalibrated pipeline. Deployed an evidence-backed, human-in-the-loop action playbook with transparent reason codes (`STALE_LOW_CTR` and `PAGE1_LOW_CTR`) that prioritizes actionable editorial reviews safely.

---
### Acknowledgments & Data Credit
Built on the FlyRank ML Internship dataset — [https://flyrank.ai](https://flyrank.ai)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.